# Transform Downloaded Regulations.gov Comments

This notebook reads the bulk-downloaded CSV files in `comments_original/`, normalizes their column names/content, and writes one processed CSV per policy into `comments/`.

The output CSVs include the columns used by the evaluation/analysis notebooks, especially `policy_id`, `source_label`, `display_name`, and `comment_text`.

In [ ]:
from pathlib import Path
import html
import re

import pandas as pd

In [ ]:
BASE_DIR = Path.cwd()
INPUT_DIR = BASE_DIR / "comments_original"
OUTPUT_DIR = BASE_DIR / "comments"

# Keep every downloaded row. Set this to an integer only if you want a smaller sample.
MAX_COMMENTS_PER_POLICY = None

SOURCE_LABEL = "human"
OUTPUT_DIR.mkdir(exist_ok=True)

# Minimal schema needed by comment_evaluator.ipynb / comment_analysis.ipynb.
EVALUATION_COLUMNS = ["source", "source_label", "policy_id", "display_name", "comment_text"]

POLICY_OUTPUT_NAMES = {
    "FDA-2024-C-1085": "fda_comments_human.csv",
    "EPA-HQ-OW-2025-2929": "epa_comments_human.csv",
    "FMCSA-2023-0257": "fmcsa_comments_human.csv",
}

In [ ]:
def clean_text(value):
    """Normalize downloaded comment text without stripping meaningful line breaks."""
    if pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = text.replace("<br/>", "\n").replace("<br>", "\n").replace("<br />", "\n")
    text = re.sub(r"\r\n?", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def first_nonempty(*values):
    for value in values:
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return ""


def build_display_name(row):
    organization = first_nonempty(row.get("Organization Name"), row.get("Government Agency"))
    if organization:
        return organization

    first_name = first_nonempty(row.get("First Name"))
    last_name = first_nonempty(row.get("Last Name"))
    full_name = " ".join(part for part in [first_name, last_name] if part).strip()
    if full_name:
        return full_name

    title = first_nonempty(row.get("Title"))
    if title.lower().startswith("comment from "):
        return title[len("comment from "):].strip()
    if title.lower().startswith("comment submitted by "):
        return title[len("comment submitted by "):].strip()
    return title


def normalize_regulations_csv(path: Path) -> pd.DataFrame:
    raw = pd.read_csv(path, dtype=str, keep_default_na=False)
    rows = []

    for idx, row in raw.iterrows():
        policy_id = first_nonempty(row.get("Docket ID"))
        comment_id = first_nonempty(row.get("Document ID"))
        comment_text = clean_text(row.get("Comment"))
        display_name = build_display_name(row)
        first_name = first_nonempty(row.get("First Name"))
        last_name = first_nonempty(row.get("Last Name"))
        organization = first_nonempty(row.get("Organization Name"), row.get("Government Agency"))
        category = first_nonempty(row.get("Category"), row.get("Government Agency Type"))

        rows.append({
            "source": f"csv:{path.name}:row{idx}",
            "source_label": SOURCE_LABEL,
            "policy_id": policy_id,
            "comment_id": comment_id,
            "request_id": comment_id or f"{policy_id}-row-{idx:05d}",
            "display_name": display_name,
            "first_name": first_name,
            "last_name": last_name,
            "organization": organization,
            "entity_type": category,
            "category": category,
            "title": first_nonempty(row.get("Title")),
            "agency_id": first_nonempty(row.get("Agency ID")),
            "document_type": first_nonempty(row.get("Document Type")),
            "document_subtype": first_nonempty(row.get("Document Subtype")),
            "comment_on_document_id": first_nonempty(row.get("Comment on Document ID")),
            "tracking_number": first_nonempty(row.get("Tracking Number")),
            "posted_date": first_nonempty(row.get("Posted Date")),
            "received_date": first_nonempty(row.get("Received Date")),
            "postmark_date": first_nonempty(row.get("Postmark Date")),
            "is_withdrawn": first_nonempty(row.get("Is Withdrawn?")),
            "duplicate_comments": first_nonempty(row.get("Duplicate Comments")),
            "comment_text": comment_text,
            "attachment_files": first_nonempty(row.get("Attachment Files")),
            "content_files": first_nonempty(row.get("Content Files")),
            "detail_url": f"https://www.regulations.gov/comment/{comment_id}" if comment_id else "",
            "status_code": 200,
        })

    df = pd.DataFrame(rows)
    df = df[df["policy_id"].astype(str).str.strip().ne("")].copy()
    df = df[df["comment_text"].astype(str).str.strip().ne("")].copy()
    return df

In [ ]:
input_files = sorted(INPUT_DIR.glob("*.csv"))
if not input_files:
    raise FileNotFoundError(f"No CSV files found in {INPUT_DIR}")

frames = []
for path in input_files:
    df = normalize_regulations_csv(path)
    print(f"{path.name}: {len(df):,} usable rows; policies={sorted(df['policy_id'].unique())}")
    frames.append(df)

all_comments = pd.concat(frames, ignore_index=True)
all_comments = all_comments.drop_duplicates(subset=["comment_id"], keep="first")

print("\nCombined rows by policy:")
display(all_comments["policy_id"].value_counts().rename_axis("policy_id").to_frame("rows"))

In [ ]:
written = []

for policy_id, group in all_comments.groupby("policy_id", sort=True):
    group = group.sort_values(["posted_date", "comment_id"], na_position="last").reset_index(drop=True)
    if MAX_COMMENTS_PER_POLICY is not None:
        group = group.head(MAX_COMMENTS_PER_POLICY).copy()

    group = group[EVALUATION_COLUMNS].copy()
    group = group.dropna(axis=1, how="all")

    output_name = POLICY_OUTPUT_NAMES.get(policy_id, f"{policy_id.lower()}_comments_human.csv")
    output_path = OUTPUT_DIR / output_name
    group.to_csv(output_path, index=False)
    written.append((policy_id, output_path, len(group)))

for policy_id, output_path, rows in written:
    print(f"{policy_id}: wrote {rows:,} rows -> {output_path}")

In [ ]:
# Sanity check: show the schema and first row from each generated CSV.
for _, output_path, _ in written:
    preview = pd.read_csv(output_path, nrows=1)
    print(f"\n{output_path.name}")
    print(list(preview.columns))
    display(preview)